In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from collections import defaultdict
import numpy as np

# ============== STEP 1: DATA TRANSFORMS ==============

# Training transform with augmentation
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Test transform (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ============== STEP 2: DATASET SPLIT FUNCTION ==============

def split_dataset_by_class(dataset, train_ratio=0.7, val_ratio=0.15):
    """Split dataset by class to prevent data leakage"""
    class_indices = defaultdict(list)
    for idx, (_, label) in enumerate(dataset.imgs):
        class_indices[label].append(idx)
    
    train_indices = []
    val_indices = []
    test_indices = []
    
    for class_label, indices in class_indices.items():
        n_samples = len(indices)
        n_train = int(train_ratio * n_samples)
        n_val = int(val_ratio * n_samples)
        
        np.random.shuffle(indices)
        
        train_indices.extend(indices[:n_train])
        val_indices.extend(indices[n_train:n_train + n_val])
        test_indices.extend(indices[n_train + n_val:])
    
    print("Dataset Split Summary:")
    print("=" * 60)
    for class_label, indices in class_indices.items():
        class_name = dataset.classes[class_label]
        n_total = len(indices)
        n_train = int(train_ratio * n_total)
        n_val = int(val_ratio * n_total)
        n_test = n_total - n_train - n_val
        print(f"{class_name:15} -> Train: {n_train:3}, Val: {n_val:3}, Test: {n_test:3}")
    print("=" * 60)
    
    return train_indices, val_indices, test_indices

# ============== STEP 3: LOAD AND SPLIT DATA ==============

dataset_path = '../../dataset/ClientFace'
full_dataset_no_transform = datasets.ImageFolder(root=dataset_path)

train_indices, val_indices, test_indices = split_dataset_by_class(
    full_dataset_no_transform,
    train_ratio=0.7,
    val_ratio=0.15
)

# Create datasets with transforms
train_dataset_full = datasets.ImageFolder(root=dataset_path, transform=train_transform)
val_dataset_full = datasets.ImageFolder(root=dataset_path, transform=test_transform)
test_dataset_full = datasets.ImageFolder(root=dataset_path, transform=test_transform)

train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(val_dataset_full, val_indices)
test_dataset = Subset(test_dataset_full, test_indices)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"\nTotal: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")

# ============== STEP 4: SETUP EFFICIENTNET V2 ==============

num_classes = len(full_dataset_no_transform.classes)
class_names = full_dataset_no_transform.classes

print(f"\nNumber of classes: {num_classes}")
print(f"Classes: {class_names}")

# Load EfficientNetV2-S (Small version)
# Options: efficientnet_v2_s, efficientnet_v2_m, efficientnet_v2_l
efficientnet_v2 = models.efficientnet_v2_s(pretrained=True)

# Check the classifier structure
print(f"\nOriginal classifier: {efficientnet_v2.classifier}")

# Replace the final layer
# EfficientNetV2 has a different classifier structure than MobileNet
num_features = efficientnet_v2.classifier[1].in_features
efficientnet_v2.classifier[1] = nn.Linear(num_features, num_classes)

print(f"Modified classifier: {efficientnet_v2.classifier}")

# Move to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
efficientnet_v2 = efficientnet_v2.to(device)

print(f"\nUsing device: {device}")

# ============== STEP 5: TRAINING SETUP ==============

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(efficientnet_v2.parameters(), lr=0.001)

# Optional: Learning rate scheduler for better convergence
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

# ============== STEP 6: TRAINING LOOP ==============

num_epochs = 20
train_losses = []
val_losses = []
train_accs = []
val_accs = []
best_val_acc = 0.0

print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60)

for epoch in range(num_epochs):
    # ========== TRAINING PHASE ==========
    efficientnet_v2.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = efficientnet_v2(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # Print progress every 10 batches
        if (batch_idx + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{len(train_loader)}], '
                  f'Loss: {loss.item():.4f}')
    
    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # ========== VALIDATION PHASE ==========
    efficientnet_v2.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = efficientnet_v2(inputs)
            loss = criterion(outputs, labels)
            
            val_running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_loss = val_running_loss / len(val_loader)
    val_acc = 100 * val_correct / val_total
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # Update learning rate scheduler
    scheduler.step(val_loss)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(efficientnet_v2.state_dict(), 'efficientnet_v2_best.pth')
        print(f'✅ New best model saved! Val Acc: {val_acc:.2f}%')
    
    print(f'Epoch {epoch+1}/{num_epochs} - '
          f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% - '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
    print("-" * 60)

# ============== STEP 7: SAVE FINAL MODEL ==============

torch.save(efficientnet_v2.state_dict(), 'efficientnet_v2_final.pth')

checkpoint = {
    'epoch': num_epochs,
    'model_state_dict': efficientnet_v2.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'num_classes': num_classes,
    'class_names': class_names,
    'best_val_acc': best_val_acc,
    'train_losses': train_losses,
    'val_losses': val_losses,
    'train_accs': train_accs,
    'val_accs': val_accs
}
torch.save(checkpoint, 'efficientnet_v2_checkpoint.pth')

print("\n✅ Training complete!")
print(f"Best validation accuracy: {best_val_acc:.2f}%")
print("Models saved:")
print("  - efficientnet_v2_best.pth (best validation accuracy)")
print("  - efficientnet_v2_final.pth (final epoch)")
print("  - efficientnet_v2_checkpoint.pth (full checkpoint)")

Dataset Split Summary:
0001            -> Train: 174, Val:  37, Test:  38
0002            -> Train:  39, Val:   8, Test:  10
0003            -> Train:  79, Val:  16, Test:  18
0004            -> Train: 476, Val: 102, Test: 103
0005            -> Train: 133, Val:  28, Test:  29
0006            -> Train: 510, Val: 109, Test: 111
0007            -> Train: 533, Val: 114, Test: 115
0008            -> Train:  86, Val:  18, Test:  19
0009            -> Train: 149, Val:  31, Test:  33
0010            -> Train:  53, Val:  11, Test:  12
0011            -> Train: 286, Val:  61, Test:  62
0012            -> Train: 304, Val:  65, Test:  66
0013            -> Train: 330, Val:  70, Test:  72
0014            -> Train: 333, Val:  71, Test:  73
0015            -> Train:  82, Val:  17, Test:  19

Total: Train=3567, Val=758, Test=780

Number of classes: 15
Classes: ['0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0014', '0015']


c:\Users\cagan\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\cagan\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_V2_S_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to C:\Users\cagan/.cache\torch\hub\checkpoints\efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:10<00:00, 8.67MB/s]



Original classifier: Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)
Modified classifier: Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=15, bias=True)
)

Using device: cuda


TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

: 